# Semana 4 · Segmentación con K-Means

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('../data/customers.csv')
print("Dimensiones del dataset:", df.shape)
df.head()

Dimensiones del dataset: (2000, 14)


,customer_id,age,city,channel_preference,annual_spend,avg_order_value,orders_last_12m,returns_ratio,bike_interest,service_interest,snapshot_date,tenure_months,days_since_last_order,returned_orders_last_12m
0,C00001,48.0,Alzira,store,4566.25,913.25,5,0.0000,electric,low,2026-06-30,67,54.0,0
1,C00002,38.0,Carcaixent,store,3835.08,639.18,6,0.0000,electric,low,2026-06-30,95,23.0,0
2,C00003,43.0,Gandia,store,1159.47,128.83,9,0.0000,urban,medium,2026-06-30,9,1.0,0
3,C00004,59.0,Carcaixent,store,2702.07,300.23,9,0.1111,electric,low,2026-06-30,79,84.0,1
4,C00005,18.0,Gandia,ecommerce,304.05,101.35,3,0.0000,electric,low,2026-06-30,89,73.0,0


## 1. Limpieza y variables
- Excluimos `customer_id` y `snapshot_date`.
- Usamos los utimos pedidos, la frecuencia, gasto y devoluciones (sin ticket medio para no duplicar peso con el gasto).

In [2]:
# 1. Comprobamos nulos
print("Valores nulos detectados:")
print(df.isna().sum())

Valores nulos detectados:
customer_id                  0
age                         20
city                         0
channel_preference           0
annual_spend                30
avg_order_value              0
orders_last_12m              0
returns_ratio                0
bike_interest                0
service_interest             0
snapshot_date                0
tenure_months                0
days_since_last_order       20
returned_orders_last_12m     0
dtype: int64


In [3]:
# Reconstruimos el gasto anual faltante (avg_order_value * orders_last_12m) y rellenamos días sin compra con la mediana
df['annual_spend'] = df['annual_spend'].fillna((df['avg_order_value'] * df['orders_last_12m']).round(2))
df['days_since_last_order'] = df['days_since_last_order'].fillna(df['days_since_last_order'].median())

# Diccionario de las 4 variables elegidas:
# - days_since_last_order : días desde la última compra (recencia)
# - orders_last_12m       : número de pedidos en el año (frecuencia)
# - annual_spend          : dinero total gastado en el año (gasto)
# - returns_ratio         : porcentaje de pedidos devueltos (devoluciones)
variables = ['days_since_last_order', 'orders_last_12m', 'annual_spend', 'returns_ratio']

# Estadísticas para comprobar valores extremos
df[variables].describe().round(2)

,days_since_last_order,orders_last_12m,annual_spend,returns_ratio
count,2000.00,2000.00,2000.00,2000.00
mean,71.55,5.25,1381.66,0.07
std,79.98,2.99,1618.98,0.13
min,0.00,1.00,35.31,0.00
25%,17.00,3.00,323.12,0.00
50%,44.00,5.00,766.71,0.00
75%,94.00,7.00,1876.86,0.10
max,364.00,17.00,13968.60,1.00


## 2. Segmentación heurística (baseline)
Reglas manuales de referencia del notebook guía.

In [4]:
def simple_segment(row):
    if pd.isna(row['annual_spend']):
        return 'revisar_dada'
    if row['annual_spend'] >= 3000 and row['returns_ratio'] <= 0.02:
        return 'premium'
    if row['annual_spend'] >= 1200 and row['orders_last_12m'] >= 8:
        return 'valor_mitja_alt'
    return 'ocasional_o_baix_valor'

df['simple_segment'] = df.apply(simple_segment, axis=1)
print("Distribución de la segmentación manual:")
print(df['simple_segment'].value_counts())
df.groupby('simple_segment')[variables].mean().round(2)

Distribución de la segmentación manual:
simple_segment
ocasional_o_baix_valor    1532
valor_mitja_alt            288
premium                    180
Name: count, dtype: int64


,days_since_last_order,orders_last_12m,annual_spend,returns_ratio
simple_segment,,,,
ocasional_o_baix_valor,80.41,4.18,808.55,0.07
premium,56.66,6.71,4808.45,0.00
valor_mitja_alt,33.73,10.02,2288.55,0.07


## 3. Escalado y elección de k
- Escalamos con `StandardScaler` (euros vs porcentajes).
- Probamos k=2 a k=6 con inercia y silueta.

In [5]:
X = df[variables]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

resum_k = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    sil = silhouette_score(X_scaled, km.labels_)
    resum_k.append({'k': k, 'Inèrcia': round(km.inertia_, 1), 'Silhouette': round(sil, 4)})

pd.DataFrame(resum_k)

,k,Inèrcia,Silhouette
0,2,5862.3,0.2786
1,3,4549.6,0.3329
2,4,3498.0,0.3451
3,5,2782.8,0.3621
4,6,2432.0,0.3702


## 4. Modelo final (k=4)
Elegimos k=4 por silueta (~0.35) y perfiles comerciales claros.

In [6]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)
print("Número de clientes por grupo:")
print(df['cluster'].value_counts().sort_index())

Número de clientes por grupo:
cluster
0    800
1    644
2    318
3    238
Name: count, dtype: int64


## 5. Resumen de perfiles

In [7]:
resum = df.groupby('cluster')[variables].mean().round(2)
resum['clientes'] = df['cluster'].value_counts().sort_index()
resum['porcentaje'] = (resum['clientes'] / len(df) * 100).round(1)

nombres = {
    0: 'Estándar Satisfecho',
    1: 'VIP / Alto Valor',
    2: 'Riesgo de Devolución',
    3: 'Inactivo / Churn'
}
resum['perfil'] = resum.index.map(nombres)
resum[['perfil', 'clientes', 'porcentaje', 'annual_spend', 'orders_last_12m', 'days_since_last_order', 'returns_ratio']]

,perfil,clientes,porcentaje,annual_spend,orders_last_12m,days_since_last_order,returns_ratio
cluster,,,,,,,
0,Estándar Satisfecho,800,40.0,693.83,3.61,51.40,0.00
1,VIP / Alto Valor,644,32.2,2866.48,8.43,38.93,0.05
2,Riesgo de Devolución,318,15.9,687.36,4.86,56.27,0.30
3,Inactivo / Churn,238,11.9,603.66,2.65,247.93,0.02


## 6. Casos límite (fronterizos)
Clientes en duda entre activo/inactivo o devolución normal/alta.

In [8]:
# 1. Clientes en duda: ¿activos o inactivos? (llevan entre 100 y 160 días sin comprar)
clientes_duda_inactivos = df[(df['days_since_last_order'] >= 100) & (df['days_since_last_order'] <= 160)]
print('Clientes en duda (activos vs inactivos, 100 a 160 días sin comprar):', len(clientes_duda_inactivos))
print(clientes_duda_inactivos[['customer_id', 'days_since_last_order', 'annual_spend', 'cluster']].head())

# 2. Clientes en duda: ¿devolución normal o problemática? (entre 8% y 18% devuelto)
clientes_duda_devolucion = df[(df['returns_ratio'] >= 0.08) & (df['returns_ratio'] <= 0.18)]
print('\nClientes en duda (devolución normal vs problemática, 8% a 18%):', len(clientes_duda_devolucion))
print(clientes_duda_devolucion[['customer_id', 'returns_ratio', 'annual_spend', 'cluster']].head())

Clientes en duda (activos vs inactivos, 100 a 160 días sin comprar): 226
   customer_id  days_since_last_order  annual_spend  cluster
7       C00008                  113.0        231.90        2
10      C00011                  139.0       1062.51        2
15      C00016                  131.0        690.92        2
22      C00023                  112.0       3558.40        1
23      C00024                  152.0        328.68        3

Clientes en duda (devolución normal vs problemática, 8% a 18%): 258
   customer_id  returns_ratio  annual_spend  cluster
3       C00004         0.1111       2702.07        1
27      C00028         0.1429        567.91        2
33      C00034         0.1667        318.60        2
34      C00035         0.1111        433.35        1
35      C00036         0.0833       2265.48        1


7. Acciones por perfil

VIP: Descuentos exclusivos en taller descuento por reparaciones, o los servicios que ofrezamos por ser cliente fiel.

Estándar: Recomendar cascos y luces, primero comprobar que hay en stock por ejemplo.

Riesgo devolución: Mejorar la guía de tallas por ejemplo que no vengan de fabrica defectuosas

Inactivos: Enviar darles un cupón descuento 

## 8. Decisión final Semana 3 (CP3)
Elegimos **Ridge** por equilibrio ($R^2 \approx 0.22$) y evitar sobreajuste antes del test.